# DNS Tunneling Detection — Deep Learning vs Classical ML (Colab)

End-to-end notebook: clone repo, install dependencies, load data, train an MLP classifier, evaluate, compare with Random Forest and XGBoost.

**Dataset**: CIC-DoHBrw-2020 flow-level features (10,000 samples, 34 numeric features)

**Repo**: https://github.com/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection  
**Open in Colab**: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection/blob/main/llm/04_distilbert_e2e.ipynb)

## How to run this notebook

> **Estimated time: ~10-15 minutes** on a free T4 GPU runtime.  
> **Optional**: T4 GPU speeds up MLP training but CPU also works.

1. Open in Colab (badge above).
2. *(Optional)* `File → Save a copy in Drive` to persist your edits.
3. Run **section 1** to clone the repo and install dependencies.
4. Run **section 2** to load and explore the dataset.
5. Run **section 3** for EDA (class balance, feature distributions).
6. Run **section 4** to scale features and split the data.
7. Run **section 5** to train the MLP deep learning model.
8. Run **section 6** to evaluate the MLP on the test set.
9. Run **section 7** to train and evaluate classical ML models (RF + XGBoost).
10. Run **section 8** for side-by-side model comparison and all plots.
11. Run **section 9** for download of results.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `RuntimeError: CUDA out of memory` | Reduce `BATCH_SIZE` to 256. |
| `ModuleNotFoundError: torch` | Re-run section 1. |
| `FileNotFoundError: .../doh_sample.csv` | Re-run section 1 to clone the repo. |
| Training is very slow | Make sure T4 GPU is enabled (Runtime → Change runtime type). |

## 1. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection.git
%cd DNS_Tunneling_Detection/llm

In [ ]:
!pip install -q scikit-learn xgboost seaborn joblib matplotlib pandas numpy tqdm torch

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## 2. Configuration and load data

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

DATA_PATH = Path("../data/sample/doh_sample.csv")
BATCH_SIZE = 512
EPOCHS = 50
LEARNING_RATE = 1e-3
OUTPUT_DIR = Path("models/mlp")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP = {"Benign": 0, "Malicious": 1}
ID2LABEL = {0: "Benign", 1: "Malicious"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.dropna()
df["Label"] = df["Label"].map(LABEL_MAP)
df = df.dropna(subset=["Label"])
df["Label"] = df["Label"].astype(int)

FEATURE_COLS = [c for c in df.select_dtypes(include="number").columns if c != "Label"]

print(f"Total samples: {len(df):,}")
print(f"Benign:    {(df['Label'] == 0).sum():,}")
print(f"Malicious: {(df['Label'] == 1).sum():,}")
print(f"Features:  {len(FEATURE_COLS)}")
print(f"Columns:   {FEATURE_COLS}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

counts = df["Label"].value_counts().rename({0: "Benign", 1: "Malicious"})
counts.plot(kind="bar", ax=axes[0], color=["#4c72b0", "#dd8452"])
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=0)

for label, name, color in [(0, "Benign", "#4c72b0"), (1, "Malicious", "#dd8452")]:
    subset = df[df["Label"] == label]
    axes[1].hist(subset["FlowBytesSent"], bins=50, alpha=0.7, label=name, color=color)
axes[1].set_title("Flow Bytes Sent Distribution")
axes[1].set_xlabel("Bytes")
axes[1].set_yscale("log")
axes[1].legend()

for label, name, color in [(0, "Benign", "#4c72b0"), (1, "Malicious", "#dd8452")]:
    subset = df[df["Label"] == label]
    axes[2].hist(subset["PacketLengthVariance"], bins=50, alpha=0.7, label=name, color=color)
axes[2].set_title("Packet Length Variance Distribution")
axes[2].set_xlabel("Variance")
axes[2].set_yscale("log")
axes[2].legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_overview.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
corr = df[FEATURE_COLS[:16]].corr()
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Heatmap (first 16 features)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "correlation_heatmap.png", dpi=150)
plt.show()

## 4. Scale features and split data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[FEATURE_COLS].values
y = df["Label"].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train: {len(X_train):,}")
print(f"Val:   {len(X_val):,}")
print(f"Test:  {len(X_test):,}")
print(f"Features: {X_train.shape[1]}")

In [ ]:
train_ds = torch.utils.data.TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
val_ds = torch.utils.data.TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long),
)
test_ds = torch.utils.data.TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long),
)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=BATCH_SIZE)

## 5. Build and train MLP classifier

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)

model = MLP(X_train.shape[1]).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

In [ ]:
from sklearn.metrics import f1_score

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
criterion = nn.CrossEntropyLoss()

history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(y_batch)
    train_loss /= len(X_train)

    model.eval()
    val_loss = 0
    val_preds = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item() * len(y_batch)
            val_preds.extend(logits.argmax(dim=-1).cpu().numpy())
    val_loss /= len(X_val)
    val_f1 = f1_score(y_val, val_preds)

    scheduler.step(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_f1={val_f1:.4f}")

model.load_state_dict(best_state)
model.to(device)
print(f"\nBest val F1: {best_val_f1:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()

axes[1].plot(history["val_f1"], label="Val F1", color="#2ca02c")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1 Score")
axes[1].set_title("Validation F1 Score")
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "mlp_training_curves.png", dpi=150)
plt.show()

In [ ]:
torch.save(model.state_dict(), OUTPUT_DIR / "mlp_model.pt")
import joblib
joblib.dump(scaler, OUTPUT_DIR / "scaler.pkl")
print(f"Model saved to {OUTPUT_DIR}")

## 6. Evaluate MLP on test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, precision_recall_curve

model.eval()
all_probs = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)

mlp_probs = np.array(all_probs)
mlp_preds = (mlp_probs >= 0.5).astype(int)

print(classification_report(y_test, mlp_preds, target_names=["Benign", "Malicious"]))

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
cm = confusion_matrix(y_test, mlp_preds)
ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malicious"]).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — MLP")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_mlp.png", dpi=150)
plt.show()

## 7. Classical ML models (Random Forest & XGBoost)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, class_weight="balanced")
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, eval_metric="logloss", tree_method="hist", n_jobs=-1, random_state=42)

rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]

xgb_preds = xgb.predict(X_test)
xgb_probs = xgb.predict_proba(X_test)[:, 1]

print("Random Forest:")
print(classification_report(y_test, rf_preds, target_names=["Benign", "Malicious"]))
print("XGBoost:")
print(classification_report(y_test, xgb_preds, target_names=["Benign", "Malicious"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

cm_rf = confusion_matrix(y_test, rf_preds)
ConfusionMatrixDisplay(cm_rf, display_labels=["Benign", "Malicious"]).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix — Random Forest")

cm_xgb = confusion_matrix(y_test, xgb_preds)
ConfusionMatrixDisplay(cm_xgb, display_labels=["Benign", "Malicious"]).plot(ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title("Confusion Matrix — XGBoost")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_classical.png", dpi=150)
plt.show()

## 8. Model comparison

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

def get_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

mlp_metrics = get_metrics(y_test, mlp_preds, mlp_probs)
rf_metrics = get_metrics(y_test, rf_preds, rf_probs)
xgb_metrics = get_metrics(y_test, xgb_preds, xgb_probs)

comparison = pd.DataFrame({
    "MLP": mlp_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics,
}).T.round(4)

comparison.to_csv("metrics_comparison.csv")
print(comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

metrics_to_plot = ["accuracy", "precision", "recall", "f1", "roc_auc"]
x = np.arange(len(metrics_to_plot))
width = 0.25
colors = ["#55a868", "#4c72b0", "#dd8452"]

for i, (name, metrics) in enumerate({"MLP": mlp_metrics, "Random Forest": rf_metrics, "XGBoost": xgb_metrics}.items()):
    vals = [metrics[m] for m in metrics_to_plot]
    ax.bar(x + i * width, vals, width, label=name, color=colors[i])

ax.set_xticks(x + width)
ax.set_xticklabels([m.replace("_", "-").upper() for m in metrics_to_plot])
ax.set_ylim(0.9, 1.01)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — All Metrics")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "metric_comparison_bars.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for name, y_true, y_prob, color in [
    ("MLP", y_test, mlp_probs, "#55a868"),
    ("Random Forest", y_test, rf_probs, "#4c72b0"),
    ("XGBoost", y_test, xgb_probs, "#dd8452"),
]:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})", color=color)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC — Model Comparison")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "roc_comparison_all.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for name, y_true, y_prob, color in [
    ("MLP", y_test, mlp_probs, "#55a868"),
    ("Random Forest", y_test, rf_probs, "#4c72b0"),
    ("XGBoost", y_test, xgb_probs, "#dd8452"),
]:
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = np.trapz(prec, rec)
    ax.plot(rec, prec, label=f"{name} (PR-AUC = {pr_auc:.4f})", color=color)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall — Model Comparison")
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "pr_comparison_all.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, model_obj, name in [(axes[0], rf, "Random Forest"), (axes[1], xgb, "XGBoost")]:
    importances = model_obj.feature_importances_
    order = np.argsort(importances)[::-1]
    top_n = 15
    sns.barplot(x=importances[order][:top_n], y=np.array(FEATURE_COLS)[order][:top_n], ax=ax, palette="viridis")
    ax.set_title(f"Top {top_n} Feature Importance — {name}")
    ax.set_xlabel("Importance")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "feature_importance_classical.png", dpi=150)
plt.show()

## 9. Download results

In [ ]:
from google.colab import files

!zip -r mlp_model.zip models/mlp/
!zip -r figures.zip figures/

print("Files ready for download:")
print("  - mlp_model.zip (trained model + scaler)")
print("  - figures.zip (all evaluation plots)")
print("  - metrics_comparison.csv (comparison table)")
print("\nDownload from the file pane on the left, or run:")
print("  files.download('mlp_model.zip')")
print("  files.download('figures.zip')")
print("  files.download('metrics_comparison.csv')")

## Summary

| Aspect | Classical ML (RF/XGBoost) | Deep Learning (MLP) |
|---|---|---|
| Input | 34 scaled numeric features | 34 scaled numeric features |
| Feature engineering | None needed (raw features) | None needed (raw features) |
| Training time | Seconds | ~1-2 min (T4) |
| Inference speed | 1000s/sec | ~1000s/sec |
| Interpretability | High (feature importance) | Low (black box) |
| Deployment | CPU-friendly | CPU-friendly (small model) |